# CS-781 Phase 3 - Rhetorical Role Prediction on our Phase-1 models
Accelerator **GPU T4 x2**, Internet On, secret `HF` (or `HF_TOKEN`). Run cells in order, or Save Version -> Save & Run All.
Everything (checkpoints, predictions, zips, summaries) is uploaded to `<you>/cs781-p1-ckpt/phase3/`. Finished runs are skipped on re-runs.

In [1]:
# ===================== P3 CELL 1 - Setup + download Phase-3 data, our tokenizer =====================
# Settings: Accelerator = GPU T4 x2, Internet = On, Secret "HF" (or "HF_TOKEN") ticked.
import os, sys, json, time, math, random, re, shutil, collections
os.environ["HF_HOME"] = "/tmp/hf_home"
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi, hf_hub_download
_sec = UserSecretsClient()
try:
    HF_TOKEN = _sec.get_secret("Phase-3")
except Exception:
    HF_TOKEN = _sec.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
HF_USER   = api.whoami()["name"]
DATA_REPO = f"{HF_USER}/cs781-p1-data"          # our tokenizer
CKPT_REPO = f"{HF_USER}/cs781-p1-ckpt"          # Phase-1 models; Phase-3 outputs go to its phase3/ folder
SRC_REPO  = "Exploration-Lab/CS-781-Capstone"
WORK      = "/kaggle/working"
P3        = "/tmp/cs781/p3"; os.makedirs(P3, exist_ok=True)

# ---- official Phase-3 files -> one flat folder ----
RAW = f"{P3}/flat"; os.makedirs(RAW, exist_ok=True)
p3_files = [x.path for x in api.list_repo_tree(SRC_REPO, repo_type="dataset", path_in_repo="Phase_3_RR", recursive=True)
            if getattr(x, "size", None) is not None]
for p in p3_files:
    lp = hf_hub_download(SRC_REPO, p, repo_type="dataset", token=HF_TOKEN, local_dir=f"{P3}/raw")
    shutil.copy(lp, f"{RAW}/{p.split('/')[-1]}")
need = ["CL_train.json", "CL_dev.json", "IT_train.json", "IT_dev.json", "CL_test.json", "IT_test.json", "label_vocab.json"]
missing = [n for n in need if not os.path.exists(f"{RAW}/{n}")]
assert not missing, f"missing files {missing} - paste this output to Claude"
print("Phase-3 files:", sorted(os.listdir(RAW)))

# ---- our tokenizer (Phase 1) ----
TOK_PATH = hf_hub_download(DATA_REPO, "tokenizer.json", repo_type="dataset", token=HF_TOKEN, local_dir=f"{P3}/tok")
from tokenizers import Tokenizer
tok = Tokenizer.from_file(TOK_PATH)
print("tokenizer vocab:", tok.get_vocab_size())

import torch
print(f"torch {torch.__version__} | GPUs {torch.cuda.device_count()}",
      *[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Phase-3 files: ['CL_dev.json', 'CL_test.json', 'CL_train.json', 'IT_dev.json', 'IT_test.json', 'IT_train.json', 'label_vocab.json']


tokenizer.json:   0%|          | 0.00/2.32M [00:00<?, ?B/s]

tokenizer vocab: 32768
torch 2.11.0+cu128 | GPUs 2 Tesla T4 Tesla T4


In [2]:
%%writefile /kaggle/working/p2text.py
"""CS-781 Phase 2 (L-NER) text utilities: loading, normalisation WITH an exact character map, token alignment,
BIO encoding/decoding. Every offset that leaves this module is an offset into the ORIGINAL document text."""
import bisect, json, unicodedata

# same punctuation map as Phase-1 pre-processing (the models were pretrained on text normalised this way)
PUNCT = {"\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'", "\u2032": "'",
         "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u2033": '"',
         "\u2010": "-", "\u2011": "-", "\u2012": "-", "\u2013": "-", "\u2014": "-",
         "\u2015": "-", "\u2212": "-", "\u2022": "*", "\u00b7": "."}


def load_texts(path):
    d = json.load(open(path, encoding="utf-8"))
    if isinstance(d, list):                                   # tolerate [{"id":..,"text":..}, ...]
        d = {str(x.get("id")): x.get("text") for x in d}
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            v = v.get("text", "")
        out[str(k)] = v if isinstance(v, str) else ""
    return out


def _ent(x):
    if isinstance(x, dict):
        s = x.get("start", x.get("start_char", x.get("begin")))
        e = x.get("end", x.get("end_char"))
        l = x.get("label", x.get("type", x.get("entity")))
        return int(s), int(e), str(l)
    return int(x[0]), int(x[1]), str(x[2])


def load_gold(path):
    d = json.load(open(path, encoding="utf-8"))
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            v = v.get("entities", v.get("spans", []))
        out[str(k)] = [_ent(x) for x in (v or [])]
    return out


def normalize_with_map(text):
    """-> (norm, omap): norm is single-spaced ASCII (like Phase-1 training text); omap[k] = index in `text` of norm[k].
    Whitespace runs become one space (mapped to the first whitespace char); non-ASCII chars are NFKC-folded or dropped."""
    out, omap, prev_space = [], [], True
    for i, ch in enumerate(text):
        c = PUNCT.get(ch, ch)
        if c.isspace():
            pass
        elif ord(c) < 32 or ord(c) == 127 or unicodedata.category(c) == "Cc":   # invisible control chars: drop
            continue
        if c.isspace():
            if not prev_space:
                out.append(" "); omap.append(i); prev_space = True
            continue
        if ord(c) > 127:
            c = "".join(x for x in unicodedata.normalize("NFKC", c) if ord(x) < 128 and not x.isspace())
            if not c:                                   # accented letters: strip the accent (e-acute -> e)
                c = "".join(x for x in unicodedata.normalize("NFKD", ch) if ord(x) < 128 and not x.isspace())
            if not c:                                   # anything else (Hindi, symbols) is dropped
                continue
        for x in c:
            out.append(x); omap.append(i)
        prev_space = False
    if out and out[-1] == " ":
        out.pop(); omap.pop()
    return "".join(out), omap


def orig_to_norm_span(omap, s, e):
    """Original [s,e) -> normalised [a,b) covering the same characters (None if nothing survives)."""
    a = bisect.bisect_left(omap, s)
    b = bisect.bisect_left(omap, e)
    return (a, b) if b > a else None


def norm_to_orig_span(omap, a, b, text=None):
    """Normalised [a,b) -> original [s,e). With `text`, the span also absorbs characters that normalisation DROPPED
    when they touch it without whitespace (e.g. a Hindi letter or a rare symbol at the end of a word)."""
    s, e = omap[a], omap[b - 1] + 1
    if text is not None:
        while e < len(text) and not text[e].isspace() and (b >= len(omap) or omap[b] != e):
            e += 1
        while s > 0 and not text[s - 1].isspace() and (a == 0 or omap[a - 1] != s - 1):
            s -= 1
    return s, e


def _is_punct(c):
    return not (c.isalnum() or c.isspace())


def tokenize(tok, norm, split_punct=True):
    """-> ids, spans: spans[i] = (a,b) normalised chars of token i WITHOUT its leading space ((a,a) if pure space).
    split_punct: BPE merges runs of punctuation into one token (e.g. ").", ".[", "),"); an entity boundary can fall
    inside such a run, so every multi-character punctuation token is split into one token per character
    (all single characters are in the byte-level vocabulary, so ids stay valid)."""
    enc = tok.encode(norm, add_special_tokens=False)
    ids, spans = [], []
    for tid, (a, b) in zip(enc.ids, enc.offsets):
        lead = a < b and norm[a] == " "
        core_a = a + 1 if lead else a
        core = norm[core_a:b]
        if split_punct and len(core) > 1 and all(_is_punct(c) for c in core):
            new_ids, new_spans = [], []
            for j, c in enumerate(core):
                piece = ("\u0120" + c) if (j == 0 and lead) else c        # \u0120 = byte-level marker for a space
                pid = tok.token_to_id(piece)
                if pid is None and j == 0 and lead:                     # "<space>X" not in vocab: emit space + X
                    sp = tok.token_to_id("\u0120")
                    if sp is not None:
                        new_ids.append(sp); new_spans.append((core_a, core_a))
                    pid = tok.token_to_id(c)
                new_ids.append(pid); new_spans.append((core_a + j, core_a + j + 1))
            if all(x is not None for x in new_ids):
                ids += new_ids; spans += new_spans
                continue                                                # else: keep the original token unsplit
        ids.append(tid); spans.append((core_a, b))
    assert all(x is not None for x in ids), "tokenizer produced an unknown id"
    return ids, spans


def spans_to_bio(tok_spans, ents_norm, label2id):
    """ents_norm: list of (a,b,label) in normalised chars. Returns tag ids: 0=O, 2k+1=B-label_k, 2k+2=I-label_k.
    A token is inside an entity if its (trimmed) chars overlap it. Overlapping entities: the earlier/longer one wins."""
    tags = [0] * len(tok_spans)
    starts = [a for a, _ in tok_spans]
    for a, b, lab in sorted(ents_norm, key=lambda x: (x[0], -(x[1] - x[0]))):
        k = label2id[lab]
        i = max(0, bisect.bisect_right(starts, a) - 1)
        first = True
        while i < len(tok_spans) and tok_spans[i][0] < b:
            ta, tb = tok_spans[i]
            if tb > ta and tb > a and ta < b and tags[i] == 0:
                tags[i] = 2 * k + 1 if first else 2 * k + 2
                first = False
            i += 1
    for i in range(1, len(tags) - 1):          # a pure-space token inside an entity continues it
        if tok_spans[i][1] <= tok_spans[i][0] and tags[i - 1] and tags[i + 1] and tags[i + 1] % 2 == 0 \
                and (tags[i + 1] - 1) // 2 == (tags[i - 1] - 1) // 2:
            tags[i] = tags[i + 1]
    return tags


def bio_to_spans(tags, tok_spans, id2label):
    """Greedy BIO decoding (an I- without a matching open entity starts a new one). -> [(a,b,label)] normalised."""
    out, cur = [], None
    for t, (a, b) in zip(tags, tok_spans):
        if b <= a:                       # pure-space token: does not close or extend anything
            continue
        if t == 0:
            if cur: out.append(cur); cur = None
            continue
        lab = id2label[(t - 1) // 2]
        is_b = (t - 1) % 2 == 0
        if is_b or cur is None or cur[2] != lab:
            if cur: out.append(cur)
            cur = [a, b, lab]
        else:
            cur[1] = b
    if cur: out.append(cur)
    return [tuple(x) for x in out]


def doc_roundtrip(tok, text, gold, label2id, id2label):
    """Gold -> tokens -> BIO -> spans -> original offsets. Returns (recovered set, gold set, n_tokens)."""
    norm, omap = normalize_with_map(text)
    ids, tsp = tokenize(tok, norm)
    ents = []
    for s, e, l in gold:
        r = orig_to_norm_span(omap, s, e)
        if r: ents.append((r[0], r[1], l))
    tags = spans_to_bio(tsp, ents, label2id)
    rec = {(*norm_to_orig_span(omap, a, b, text), l) for a, b, l in bio_to_spans(tags, tsp, id2label)}
    return rec, {(s, e, l) for s, e, l in gold}, len(ids)


def strict_macro(truth, pred, labels):
    """Same computation as evaluate_phase2.py. truth/pred: {doc: iterable of (s,e,label)}."""
    st = {l: [0, 0, 0] for l in labels}
    for d in truth:
        g = set(map(tuple, truth[d])); p = set(map(tuple, pred.get(d, [])))
        for l in labels:
            gs = {x for x in g if x[2] == l}; ps = {x for x in p if x[2] == l}
            st[l][0] += len(gs & ps); st[l][1] += len(ps - gs); st[l][2] += len(gs - ps)
    per = {}
    for l in labels:
        tp, fp, fn = st[l]
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[l] = (pr, rc, 2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    n = len(labels)
    return {"f1": sum(v[2] for v in per.values()) / n, "p": sum(v[0] for v in per.values()) / n,
            "r": sum(v[1] for v in per.values()) / n, "per_label": per}


Writing /kaggle/working/p2text.py


In [3]:
%%writefile /kaggle/working/model.py
"""CS-781 Phase 1 - four legal language-model architectures, written from scratch in PyTorch.

  gpt2    : classic decoder-only Transformer (learned absolute positions, LayerNorm, GELU MLP, multi-head attention)
  llama   : modern decoder-only Transformer (RoPE, RMSNorm, SwiGLU, grouped-query attention, no biases)
  shared  : lightweight decoder (ALBERT-style factorised embeddings + cycle cross-layer weight sharing + multi-query attn)
  encdec  : encoder-decoder Transformer (bidirectional encoder over the prompt, causal decoder with cross-attention)

All models: tied input/output embeddings, pre-norm residual blocks, PyTorch SDPA attention, KV-cache generation.
"""
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    arch: str = "llama"
    vocab_size: int = 32768
    d_model: int = 768
    n_layer: int = 12            # decoder-only: layers; encdec: layers in EACH of encoder and decoder; shared: effective depth
    n_head: int = 12
    n_kv_head: int = 4           # GQA groups (llama/encdec); shared uses 1 (MQA); gpt2 uses n_head (MHA)
    d_ff: int = 2304
    max_seq_len: int = 1024
    rope_theta: float = 10000.0
    dropout: float = 0.0
    n_unique_layers: int = 6     # shared only: number of distinct blocks, reused cyclically
    d_embed: int = 256           # shared only: factorised embedding size
    norm_eps: float = 1e-5

    def to_dict(self):
        return asdict(self)


def preset(arch: str, **overrides) -> ModelConfig:
    base = {
        "gpt2":   dict(arch="gpt2", d_model=768, n_layer=12, n_head=12, n_kv_head=12, d_ff=3072),
        "llama":  dict(arch="llama", d_model=768, n_layer=12, n_head=12, n_kv_head=4, d_ff=2304),
        "shared": dict(arch="shared", d_model=768, n_layer=12, n_head=12, n_kv_head=1, d_ff=3072,
                       n_unique_layers=6, d_embed=256),
        "encdec": dict(arch="encdec", d_model=768, n_layer=6, n_head=12, n_kv_head=4, d_ff=2304),
    }[arch]
    base.update(overrides)
    return ModelConfig(**base)


# ----------------------------------------------------------------------------------------------------------------
# building blocks
# ----------------------------------------------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).to(x.dtype)


class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.ln = nn.LayerNorm(d, eps=eps)

    def forward(self, x):
        return F.layer_norm(x.float(), self.ln.normalized_shape, self.ln.weight.float(), self.ln.bias.float(),
                            self.ln.eps).to(x.dtype)


def rope_cache(head_dim, max_len, theta, device=None):
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_len, device=device).float()
    freqs = torch.outer(t, inv)                       # (T, hd/2)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    # x: (B, H, T, hd); cos/sin: (T, hd/2) - rotate pairs (x[..., :hd/2], x[..., hd/2:])
    xf = x.float()
    x1, x2 = xf[..., : xf.shape[-1] // 2], xf[..., xf.shape[-1] // 2:]
    cos, sin = cos[None, None], sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.to(x.dtype)


class Attention(nn.Module):
    """Multi-head / grouped-query / multi-query attention. Self-attention (optionally causal) or cross-attention."""

    def __init__(self, cfg: ModelConfig, n_kv_head: int, bias: bool, use_rope: bool, cross: bool = False):
        super().__init__()
        d, h = cfg.d_model, cfg.n_head
        assert d % h == 0 and h % n_kv_head == 0
        self.h, self.kv, self.hd = h, n_kv_head, d // h
        self.use_rope, self.cross = use_rope, cross
        self.q = nn.Linear(d, h * self.hd, bias=bias)
        self.k = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.v = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.o = nn.Linear(h * self.hd, d, bias=bias)
        self.o.is_residual_proj = True
        self.dropout = cfg.dropout

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.hd).transpose(1, 2)          # (B, n, T, hd)

    def _expand(self, x):
        return x if self.kv == self.h else x.repeat_interleave(self.h // self.kv, dim=1)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None):
        """
        x: (B, T, d).  rope: (cos, sin) full tables.  pos: absolute position of x[:,0] (for RoPE during decoding).
        cache: dict used during generation. Self-attn: keys 'k','v' accumulate. Cross-attn: 'k','v' computed once.
        memory: encoder output (B, S, d) for cross-attention.
        """
        q = self._split(self.q(x), self.h)
        if self.cross:
            if cache is not None and "k" in cache:
                k, v = cache["k"], cache["v"]
            else:
                k, v = self._split(self.k(memory), self.kv), self._split(self.v(memory), self.kv)
                if cache is not None:
                    cache["k"], cache["v"] = k, v
            is_causal = False
        else:
            k, v = self._split(self.k(x), self.kv), self._split(self.v(x), self.kv)
            if self.use_rope:
                cos, sin = rope
                T = x.shape[1]
                q = apply_rope(q, cos[pos:pos + T], sin[pos:pos + T])
                k = apply_rope(k, cos[pos:pos + T], sin[pos:pos + T])
            if cache is not None:
                if "k" in cache:
                    k = torch.cat([cache["k"], k], dim=2); v = torch.cat([cache["v"], v], dim=2)
                cache["k"], cache["v"] = k, v
            # causal mask only needed when queries and keys cover the same span (training / prefill)
            is_causal = causal and q.shape[2] == k.shape[2] and q.shape[2] > 1
            if causal and q.shape[2] != k.shape[2] and q.shape[2] != 1:
                raise ValueError("multi-token decode with a non-empty cache is not supported")
        y = F.scaled_dot_product_attention(q, self._expand(k), self._expand(v), is_causal=is_causal,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(x.shape[0], x.shape[1], self.h * self.hd)
        return self.o(y)


class MLP(nn.Module):
    def __init__(self, cfg: ModelConfig, kind: str, bias: bool):
        super().__init__()
        self.kind = kind
        if kind == "swiglu":
            self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        else:
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=bias)
        self.down.is_residual_proj = True
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        if self.kind == "swiglu":
            h = F.silu(self.gate(x)) * self.up(x)
        else:
            h = F.gelu(self.up(x), approximate="tanh")
        return self.drop(self.down(h))


class Block(nn.Module):
    """Pre-norm Transformer block: x + Attn(Norm(x)) [+ CrossAttn(Norm(x), memory)] + MLP(Norm(x))."""

    def __init__(self, cfg: ModelConfig, style: str, n_kv_head: int, cross: bool = False):
        super().__init__()
        gpt2 = style == "gpt2"
        Norm = LayerNorm if gpt2 else RMSNorm
        self.n1 = Norm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg, n_kv_head, bias=gpt2, use_rope=not gpt2)
        self.has_cross = cross
        if cross:
            self.nc = Norm(cfg.d_model, cfg.norm_eps)
            self.xattn = Attention(cfg, n_kv_head, bias=False, use_rope=False, cross=True)
        self.n2 = Norm(cfg.d_model, cfg.norm_eps)
        self.mlp = MLP(cfg, "gelu" if gpt2 else "swiglu", bias=gpt2)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None, xcache=None):
        x = x + self.drop(self.attn(self.n1(x), rope=rope, pos=pos, causal=causal, cache=cache))
        if self.has_cross:
            x = x + self.drop(self.xattn(self.nc(x), memory=memory, cache=xcache))
        return x + self.mlp(self.n2(x))


# ----------------------------------------------------------------------------------------------------------------
# the language model
# ----------------------------------------------------------------------------------------------------------------
class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        a = cfg.arch
        assert a in ("gpt2", "llama", "shared", "encdec"), a
        d = cfg.d_model
        if a == "shared":
            self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_embed)
            self.emb_proj = nn.Linear(cfg.d_embed, d, bias=False)          # E -> d   (factorised embedding)
        else:
            self.tok_emb = nn.Embedding(cfg.vocab_size, d)
        if a == "gpt2":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, d)
        self.drop = nn.Dropout(cfg.dropout)

        if a == "gpt2":
            self.blocks = nn.ModuleList([Block(cfg, "gpt2", cfg.n_head) for _ in range(cfg.n_layer)])
        elif a == "llama":
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
        elif a == "shared":
            assert cfg.n_layer % cfg.n_unique_layers == 0
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_unique_layers)])
        else:  # encdec
            self.enc_blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
            self.enc_norm = RMSNorm(d, cfg.norm_eps)
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head, cross=True) for _ in range(cfg.n_layer)])
        self.norm = LayerNorm(d, cfg.norm_eps) if a == "gpt2" else RMSNorm(d, cfg.norm_eps)

        if a != "gpt2":
            hd = d // cfg.n_head
            cos, sin = rope_cache(hd, max(2 * cfg.max_seq_len, 2048), cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        n_res = 2 * self.n_layer_applications() + (cfg.n_layer if a == "encdec" else 0)
        for name, p in self.named_parameters():
            if name.endswith("o.weight") or name.endswith("down.weight"):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    # -- helpers ------------------------------------------------------------------------------------------------
    def n_layer_applications(self):
        return self.cfg.n_layer * (2 if self.cfg.arch == "encdec" else 1)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def _rope(self):
        return None if self.cfg.arch == "gpt2" else (self.rope_cos, self.rope_sin)

    def _embed(self, idx, pos=0):
        x = self.tok_emb(idx)
        if self.cfg.arch == "shared":
            x = self.emb_proj(x)
        if self.cfg.arch == "gpt2":
            T = idx.shape[1]
            if pos + T > self.cfg.max_seq_len:
                raise ValueError(f"gpt2 positions exceed max_seq_len ({pos + T} > {self.cfg.max_seq_len})")
            x = x + self.pos_emb(torch.arange(pos, pos + T, device=idx.device))[None]
        return self.drop(x)

    def _logits(self, h):
        h = self.norm(h)
        if self.cfg.arch == "shared":
            h = F.linear(h, self.emb_proj.weight.t())     # d -> E   (tied with the input projection)
        return F.linear(h, self.tok_emb.weight)            # tied output embedding

    def _decoder_layers(self):
        if self.cfg.arch == "shared":                       # cycle sharing: 0,1,..,5,0,1,..,5
            return [self.blocks[i % self.cfg.n_unique_layers] for i in range(self.cfg.n_layer)]
        return list(self.blocks)

    def encode(self, src):
        x = self._embed(src)
        for blk in self.enc_blocks:
            x = blk(x, rope=self._rope(), pos=0, causal=False)
        return self.enc_norm(x)

    # -- training forward -----------------------------------------------------------------------------------------
    def forward(self, idx, targets=None, src=None, loss_mask=None):
        """Decoder-only: idx (B,T) -> logits; loss vs targets (B,T).  encdec: src (B,S) is the encoder input."""
        memory = self.encode(src) if self.cfg.arch == "encdec" else None
        x = self._embed(idx)
        for blk in self._decoder_layers():
            x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
        logits = self._logits(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), reduction="none")
        if loss_mask is not None:
            m = loss_mask.reshape(-1).float()
            return logits, (loss * m).sum() / m.sum().clamp(min=1)
        return logits, loss.mean()

    # -- generation -----------------------------------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, prompt, max_new_tokens, temperature=1.0, top_k=0, top_p=1.0, repetition_penalty=1.0,
                 no_repeat_ngram=0, banned_ids=(), bos_id=2, generator=None, use_cache=True):
        """prompt: (B, T) LongTensor, all rows the same length (no padding). Returns (B, max_new_tokens) new ids.
        temperature 0 = greedy."""
        B = prompt.shape[0]
        dev = prompt.device
        layers = self._decoder_layers()
        if self.cfg.arch == "encdec":
            memory = self.encode(prompt)
            seq = torch.full((B, 1), bos_id, dtype=torch.long, device=dev)   # decoder starts from <|bos|>
            history = prompt                                                   # for repetition penalties
        else:
            memory, seq, history = None, prompt, prompt
        caches = [dict() for _ in layers] if use_cache else None
        xcaches = [dict() for _ in layers] if (use_cache and memory is not None) else [None] * len(layers)
        out = []
        cur, pos = seq, 0
        for step in range(max_new_tokens):
            if use_cache:
                x = self._embed(cur, pos=pos)
                for li, blk in enumerate(layers):
                    x = blk(x, rope=self._rope(), pos=pos, causal=True, cache=caches[li], memory=memory,
                            xcache=xcaches[li])
                pos += cur.shape[1]
            else:
                x = self._embed(seq)
                for blk in layers:
                    x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
            logits = self._logits(x[:, -1:, :])[:, -1, :].float()
            full_hist = torch.cat([history] + out, dim=1) if out else history
            nxt = sample_next(logits, full_hist, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram,
                              banned_ids, generator)
            out.append(nxt)
            seq = torch.cat([seq, nxt], dim=1)
            cur = nxt
        return torch.cat(out, dim=1)


def sample_next(logits, history, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram, banned_ids,
                generator=None):
    """logits (B,V) float32, history (B,L) long (prompt + generated so far). Returns (B,1)."""
    logits = logits.clone()
    if banned_ids:
        logits[:, list(banned_ids)] = -float("inf")
    if repetition_penalty != 1.0:
        # CTRL-style penalty on tokens already present in the recent history (last 256 tokens)
        recent = history[:, -256:]
        sc = torch.gather(logits, 1, recent)
        sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
        logits.scatter_(1, recent, sc)
    if no_repeat_ngram and history.shape[1] >= no_repeat_ngram:
        n = no_repeat_ngram
        hist = history[:, -512:].tolist()
        for b, h in enumerate(hist):
            prefix = tuple(h[-(n - 1):]) if n > 1 else tuple()
            banned = {tuple(h[i:i + n])[-1] for i in range(len(h) - n + 1) if tuple(h[i:i + n - 1]) == prefix}
            if banned:
                logits[b, list(banned)] = -float("inf")
    if temperature <= 0:
        return logits.argmax(dim=-1, keepdim=True)
    logits = logits / temperature
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.shape[-1]), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cp = torch.softmax(sl, dim=-1).cumsum(dim=-1)
        remove = cp - torch.softmax(sl, dim=-1) > top_p          # keep the smallest set with mass >= top_p
        sl = sl.masked_fill(remove, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, si, sl)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1, generator=generator)


Writing /kaggle/working/model.py


In [4]:
%%writefile /kaggle/working/p3rr.py
"""CS-781 Phase 3: Rhetorical Role prediction (one label per sentence) on top of our own Phase-1 language models.

One script for every experiment:
  --method full    : full fine-tuning (backbone + head)
  --method head    : frozen backbone, only the head is trained (linear probe)
  --method lora    : LoRA      W x + (alpha/r) * B(A x)                       (own implementation)
  --method rslora  : rsLoRA    W x + (alpha/sqrt(r)) * B(A x)   rank-stabilised scaling (own implementation)
  --method lorma   : LoRMA     (I + (alpha/r) B A) W x  = y + s*B(A y)       (own implementation, additive identity)
  --method ia3     : IA3       learned vectors rescale keys, values and the FFN hidden layer (own implementation)
  --kd_teacher R   : knowledge distillation from Phase-3 run R (its best.pt) into this (student) model
Context:
  --ctx sent       : every sentence encoded ALONE (sentence-only model)
  --ctx win        : sentences encoded inside a window of the document (token-level context)
  --head bilstm    : + a BiLSTM over the sentence vectors of the window (document-level sentence context)
Sentence vector = mean of its token states.  Long documents: sliding windows, each sentence's probabilities are
averaged over the windows that contain it (weighted by how much of the sentence is inside and how central it is).
"""
import argparse, json, math, os, random, sys, time, zipfile
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from model import LegalLM, ModelConfig                      # our Phase-1 model code (unchanged)
import p2text as PT                                          # text normalisation + tokenisation (from Phase 2)

DOMAINS = ("CL", "IT")


# ================================================================================================================
# parameter-efficient adapters (written from scratch)
# ================================================================================================================
class LoRALinear(nn.Module):
    """y = W0 x + b + scale * B(A(dropout(x))).  LoRA: scale = alpha/r ; rsLoRA: scale = alpha/sqrt(r). B = 0 at start."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, dropout=0.05, rs=False):
        super().__init__()
        self.base = base
        self.r, self.scale = r, (alpha / math.sqrt(r) if rs else alpha / r)
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA (Bihany, Patel & Modi, 2025), additive rank inflation: W' = (I + (alpha/r) B A) W0.
    y0 = W0 x ; y = y0 + s * B(A y0) (+ bias).  B = 0 -> starts exactly at W0."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        d_out = base.out_features
        self.A = nn.Parameter(torch.randn(r, d_out) / math.sqrt(d_out))
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.scale * F.linear(F.linear(self.drop(y0), self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias


class IA3Out(nn.Module):
    """IA3 (Liu et al., 2022) on keys / values: y = W x * l   (l: learned vector, starts at 1)."""

    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class IA3In(nn.Module):
    """IA3 on the feed-forward hidden layer: y = W (x * l)   (l rescales the FFN activations, starts at 1)."""

    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


def add_adapters(root: nn.Module, kind, targets, r, alpha, dropout):
    """Wrap every nn.Linear inside `root` whose attribute name is a target. IA3 always uses k, v (attention) + down (FFN)."""
    n = 0
    for mod in list(root.modules()):
        for name, child in list(mod.named_children()):
            if not isinstance(child, nn.Linear):
                continue
            if kind == "ia3":
                if name in ("k", "v"):
                    setattr(mod, name, IA3Out(child)); n += 1
                elif name == "down":
                    setattr(mod, name, IA3In(child)); n += 1
            elif name in targets:
                if kind == "lorma":
                    setattr(mod, name, LoRMALinear(child, r, alpha, dropout))
                else:
                    setattr(mod, name, LoRALinear(child, r, alpha, dropout, rs=(kind == "rslora")))
                n += 1
    return n


# ================================================================================================================
# the RR network: Phase-1 backbone -> sentence vectors (mean of tokens) -> [BiLSTM over sentences] -> 13 classes
# ================================================================================================================
class RRNet(nn.Module):
    def __init__(self, lm: LegalLM, n_cls, bidir=True, head="linear", dropout=0.1):
        super().__init__()
        self.lm, self.arch, self.bidir = lm, lm.cfg.arch, bidir
        d = lm.cfg.d_model
        if self.arch == "encdec":                      # only the (bidirectional) encoder is used
            lm.blocks = nn.ModuleList()
        self.drop = nn.Dropout(dropout)
        self.head_kind = head
        if head == "bilstm":
            self.rnn = nn.LSTM(d, 256, batch_first=True, bidirectional=True)
            self.out = nn.Linear(512, n_cls)
        elif head == "doclstm":                        # document level: 2-layer BiLSTM over ALL sentences + position
            self.rnn = nn.LSTM(d + 3, 256, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
            self.out = nn.Linear(512, n_cls)
        else:
            self.out = nn.Linear(d, n_cls)

    def features(self, idx):
        lm = self.lm
        if self.arch == "encdec":
            return lm.encode(idx)
        x = lm._embed(idx)
        for blk in lm._decoder_layers():
            x = blk(x, rope=lm._rope(), pos=0, causal=not self.bidir)
        return lm.norm(x)

    def forward(self, idx, seg, nslots, lens):
        """idx, seg: (B,T); seg = sentence slot of each token (0..lens[b]-1). Returns logits (B, nslots, C)."""
        h = self.drop(self.features(idx)).float()                       # (B,T,d)
        oh = F.one_hot(seg, nslots).float()                             # (B,T,S)
        cnt = oh.sum(1).clamp(min=1.0)                                  # (B,S)
        sv = torch.bmm(oh.transpose(1, 2), h) / cnt[..., None]          # (B,S,d) mean of each sentence's tokens
        if self.head_kind == "bilstm":
            packed = nn.utils.rnn.pack_padded_sequence(sv, lens.cpu(), batch_first=True, enforce_sorted=False)
            out, _ = self.rnn(packed)
            sv, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=nslots)
        return self.out(sv).float()


def count_params(m):
    tot = sum(p.numel() for p in m.parameters())
    tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return tot, tr


def build_net(lm_cfg, n_cls, a):
    lm = LegalLM(ModelConfig(**lm_cfg))
    return RRNet(lm, n_cls, bidir=bool(a["bidir"]), head=a["head"])


def setup_method(net, a):
    for p in net.parameters():
        p.requires_grad = True
    m = a["method"]
    if m in ("head", "lora", "rslora", "lorma", "ia3"):
        for p in net.lm.parameters():
            p.requires_grad = False
    if m in ("lora", "rslora", "lorma", "ia3"):
        root = net.lm.enc_blocks if net.arch == "encdec" else net.lm.blocks
        n = add_adapters(root, m, set(a["targets"].split(",")), a["rank"], a["alpha"], 0.05)
        print(f"{m}: wrapped {n} linear layers", flush=True)


# ================================================================================================================
# data
# ================================================================================================================
def load_vocab(path):
    v = json.load(open(path, encoding="utf-8"))
    if isinstance(v, dict):
        return [k for k, _ in sorted(v.items(), key=lambda kv: int(kv[1]))]
    return list(v)


def load_docs(path, labels_list):
    """Accepts [{id, text:[...], labels:[...]}, ...]  or  {id: [sentences]}  or  {id: {text, labels}}."""
    d = json.load(open(path, encoding="utf-8"))
    items = []
    if isinstance(d, dict):
        for k, v in d.items():
            if isinstance(v, dict):
                items.append((str(k), v.get("text", v.get("sentences")), v.get("labels")))
            else:
                items.append((str(k), v, None))
    else:
        for e in d:
            items.append((str(e["id"]), e.get("text", e.get("sentences")), e.get("labels")))
    out = []
    for did, sents, labs in items:
        assert isinstance(sents, list), f"{path}: document {did} has no sentence list"
        sents = [s if isinstance(s, str) else str(s) for s in sents]
        if labs is not None:
            assert len(labs) == len(sents), f"{path}: {did} has {len(sents)} sentences but {len(labs)} labels"
            labs = [labels_list[int(x)] if not isinstance(x, str) else x for x in labs]
            bad = [x for x in labs if x not in labels_list]
            assert not bad, f"{path}: unknown labels {bad[:3]}"
        out.append((did, sents, labs))
    return out


class Doc:
    def __init__(self, did, domain, sents, labels, tok, L2I, max_sent_tok):
        self.id, self.domain, self.sents = did, domain, sents
        self.n = len(sents)
        self.y = [L2I[l] for l in labels] if labels is not None else None
        self.ids, self.seg, self.sent_tok = [], [], []
        self.sent_ids = []
        for i, s in enumerate(sents):
            norm = PT.normalize_with_map(s)[0].strip() or "."
            ids, _ = PT.tokenize(tok, " " + norm)
            if not ids:
                ids, _ = PT.tokenize(tok, " .")
            if len(ids) > max_sent_tok:                       # very long sentence: keep its start and its end
                keep_tail = max_sent_tok // 4
                ids = ids[:max_sent_tok - keep_tail] + ids[-keep_tail:]
            self.sent_ids.append(ids)
            self.sent_tok.append(len(ids))
            self.ids += ids
            self.seg += [i] * len(ids)


def windows(n, W, stride):
    if n == 0:
        return []
    if n <= W:
        return [(0, n)]
    starts = list(range(0, n - W + 1, stride))
    if starts[-1] != n - W:
        starts.append(n - W)
    return [(s, s + W) for s in starts]


def make_item(doc, kind, a, b):
    """kind 'win': tokens a..b of the document; kind 'sent': sentence a alone.
    Returns ids, local seg, list of (sentence index, weight = share of the sentence inside, centre of its tokens)."""
    if kind == "sent":
        ids = doc.sent_ids[a]
        return ids, [0] * len(ids), [(a, 1.0, 0.5)]
    ids = doc.ids[a:b]
    gseg = doc.seg[a:b]
    first = gseg[0]
    seg = [s - first for s in gseg]
    slots = []
    L = len(ids)
    pos = defaultdict(list)
    for j, s in enumerate(gseg):
        pos[s].append(j)
    for s in range(first, gseg[-1] + 1):
        p = pos[s]
        slots.append((s, len(p) / doc.sent_tok[s], (np.mean(p) + 0.5) / L))
    return ids, seg, slots


def train_items(docs, ctx, W, rng):
    items = []
    for di, d in enumerate(docs):
        if d.n == 0:
            continue
        if ctx == "sent":
            items += [(di, "sent", i, None) for i in range(d.n)]
            continue
        n = len(d.ids)
        if n <= W:
            items.append((di, "win", 0, n)); continue
        off = rng.randrange(0, W // 2)
        s = off
        while s + W <= n:
            items.append((di, "win", s, s + W)); s += W
        items.append((di, "win", n - W, n))
        if off > 0:
            items.append((di, "win", 0, W))
    return items


def batches_by_length(items, docs, bs, rng):
    """Group items of equal token length (no padding is ever needed: our Phase-1 models have no padding mask)."""
    rng.shuffle(items)
    by_len = defaultdict(list)
    for it in items:
        di, kind, a, b = it
        L = docs[di].sent_tok[a] if kind == "sent" else b - a
        by_len[L].append(it)
    out = []
    for L, its in by_len.items():
        for i in range(0, len(its), bs):
            out.append(its[i:i + bs])
    rng.shuffle(out)
    return out


def collate(batch, docs, device):
    ids, segs, slots = [], [], []
    for di, kind, a, b in batch:
        i_, s_, sl = make_item(docs[di], kind, a, b)
        ids.append(i_); segs.append(s_); slots.append(sl)
    S = max(len(s) for s in slots)
    x = torch.tensor(ids, dtype=torch.long, device=device)
    seg = torch.tensor(segs, dtype=torch.long, device=device)
    lens = torch.tensor([len(s) for s in slots], dtype=torch.long)
    return x, seg, S, lens, slots


def doc_logits(net, doc, W, off, device, amp, grad_win=None, bs=16):
    """Whole-document model: the document is cut into consecutive windows of W tokens (first cut at `off`);
    every window goes through the Phase-1 backbone (only windows grad_win=[g0,g1) keep gradients - memory),
    token states are averaged per sentence, + [relative position, its square, domain flag], then a 2-layer
    BiLSTM reads ALL sentences of the document in order. Returns logits (n_sentences, C)."""
    n = len(doc.ids)
    cuts = sorted(set([0] + [c for c in range(off % W if off else W, n, W)] + [n]))
    wins = list(zip(cuts[:-1], cuts[1:]))
    feats = [None] * len(wins)
    g0, g1 = grad_win if grad_win else (0, 0)

    def run(idx, grad):
        groups = defaultdict(list)
        for i in idx:
            groups[wins[i][1] - wins[i][0]].append(i)
        for L, ii in groups.items():
            for k in range(0, len(ii), bs):
                part = ii[k:k + bs]
                x = torch.tensor([doc.ids[wins[i][0]:wins[i][1]] for i in part], dtype=torch.long, device=device)
                with torch.set_grad_enabled(grad), amp():
                    h = net.features(x).float()
                for r, i in enumerate(part):
                    feats[i] = h[r]
    run([i for i in range(len(wins)) if g0 <= i < g1], True)
    run([i for i in range(len(wins)) if not (g0 <= i < g1)], False)
    H = net.drop(torch.cat(feats, 0))                                            # (n_tokens, d)
    seg = torch.tensor(doc.seg, dtype=torch.long, device=device)
    sv = torch.zeros(doc.n, H.shape[1], device=device).index_add(0, seg, H)
    cnt = torch.zeros(doc.n, device=device).index_add(0, seg, torch.ones_like(seg, dtype=torch.float))
    sv = sv / cnt.clamp(min=1.0)[:, None]
    rel = torch.arange(doc.n, device=device, dtype=torch.float) / max(1, doc.n - 1)
    dom = torch.full_like(rel, 1.0 if doc.domain == "IT" else 0.0)
    x = torch.cat([sv, torch.stack([rel, rel * rel, dom], -1)], -1)[None]
    with amp():
        out, _ = net.rnn(x)
        return net.out(out[0]).float()


@torch.no_grad()
def predict_probs(net, doc, ctx, W, stride, device, amp, bs=16):
    """-> (n_sentences, C) probabilities."""
    C = net.out.out_features
    if doc.n == 0:
        return np.zeros((0, C))
    if ctx == "doc":                                   # two cuttings of the document (offset 0 and W/2), averaged
        p = sum(torch.softmax(doc_logits(net, doc, W, off, device, amp), -1) for off in (0, W // 2)) / 2
        return p.cpu().numpy().astype(np.float64)
    if ctx == "sent":
        items = [(0, "sent", i, None) for i in range(doc.n)]
    else:
        items = [(0, "win", a, b) for a, b in windows(len(doc.ids), W, stride)]
    groups = defaultdict(list)
    for it in items:
        L = doc.sent_tok[it[2]] if it[1] == "sent" else it[3] - it[2]
        groups[L].append(it)
    acc = np.zeros((doc.n, C)); wsum = np.zeros(doc.n)
    for L, its in groups.items():
        for i in range(0, len(its), bs):
            x, seg, S, lens, slots = collate(its[i:i + bs], [doc], device)
            with amp():
                logits = net(x, seg, S, lens)
            probs = torch.softmax(logits.float(), -1).cpu().numpy()
            for r, sl in enumerate(slots):
                for k, (s, share, centre) in enumerate(sl):
                    w = share * (1.0 - abs(centre - 0.5) * 1.8)          # inside share x centrality (1.0 .. 0.1)
                    acc[s] += w * probs[r, k]; wsum[s] += w
    assert (wsum > 0).all(), "a sentence was not covered by any window"
    return acc / wsum[:, None]


# ================================================================================================================
# metric (macro-F1 over sentence labels; labels present in truth or prediction, as sklearn's default)
# ================================================================================================================
def macro_scores(truth, pred, labels):
    """EXACTLY the official evaluate_phase3.py metric: sklearn f1/precision/recall with labels=<all 13>,
    average='macro', zero_division=0 (a label that is never predicted correctly - or absent - counts as 0).
    truth/pred: dict doc -> list of labels. Also returns f1 over present labels (info), accuracy, per-label."""
    t = [l for d in truth for l in truth[d]]
    p = [l for d in truth for l in pred[d]]
    assert len(t) == len(p)
    per = {}
    present = []
    for l in labels:
        tp = sum(1 for a, b in zip(t, p) if a == l and b == l)
        fp = sum(1 for a, b in zip(t, p) if a != l and b == l)
        fn = sum(1 for a, b in zip(t, p) if a == l and b != l)
        P = tp / (tp + fp) if tp + fp else 0.0
        R = tp / (tp + fn) if tp + fn else 0.0
        F_ = 2 * P * R / (P + R) if P + R else 0.0
        per[l] = [P, R, F_, tp + fn]
        if tp + fn + fp > 0:
            present.append(l)
    return {"f1": float(np.mean([per[l][2] for l in labels])),
            "p": float(np.mean([per[l][0] for l in labels])),
            "r": float(np.mean([per[l][1] for l in labels])),
            "f1_present": float(np.mean([per[l][2] for l in present])) if present else 0.0,
            "acc": float(np.mean([a == b for a, b in zip(t, p)])) if t else 0.0,
            "per_label": {l: [round(v, 4) for v in per[l][:3]] + [per[l][3]] for l in labels}}


TAUS = (0.0, 0.25, 0.5, 0.75, 1.0)


def prior_adjust(probs, prior, tau):
    """Macro-F1 calibration: p(c) / prior(c)^tau, renormalised. tau=0: unchanged; tau=1: fully prior-corrected
    (rare roles get the same chance as frequent ones). ONE number, chosen on dev."""
    if tau == 0:
        return probs
    q = probs / (prior[None, :] ** tau)
    return q / q.sum(-1, keepdims=True)


def train_prior(docs, C):
    cnt = np.bincount([y for d in docs for y in d.y], minlength=C).astype(float) + 1.0
    return cnt / cnt.sum()


# ================================================================================================================
def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--run_name", required=True)
    p.add_argument("--base_run", required=True, help="Phase-1 run, e.g. dapt_llama / explore_gpt2")
    p.add_argument("--base_file", default="best_model.pt")
    p.add_argument("--method", default="lorma", choices=["full", "head", "lora", "rslora", "lorma", "ia3"])
    p.add_argument("--kd_teacher", default="", help="Phase-3 run name of the teacher (enables distillation)")
    p.add_argument("--kd_alpha", type=float, default=0.5, help="weight of the gold-label loss (1-alpha: teacher)")
    p.add_argument("--kd_T", type=float, default=2.0)
    p.add_argument("--bidir", type=int, default=1, help="1 = bidirectional attention in decoder backbones")
    p.add_argument("--ctx", default="win", choices=["sent", "win", "doc"])
    p.add_argument("--grad_windows", type=int, default=12, help="ctx doc: windows per document that keep gradients")
    p.add_argument("--head", default="bilstm", choices=["linear", "bilstm"])
    p.add_argument("--rank", type=int, default=16)
    p.add_argument("--alpha", type=float, default=32)
    p.add_argument("--targets", default="q,k,v,o")
    p.add_argument("--win", type=int, default=512, help="training window (tokens)")
    p.add_argument("--infer_win", type=int, default=512)
    p.add_argument("--infer_stride", type=int, default=256)
    p.add_argument("--max_sent_tok", type=int, default=256)
    p.add_argument("--epochs", type=int, default=15)
    p.add_argument("--bs", type=int, default=16)
    p.add_argument("--lr", type=float, default=5e-5, help="backbone LR (full)")
    p.add_argument("--peft_lr", type=float, default=1e-3, help="adapter LR (lora / rslora / lorma / ia3)")
    p.add_argument("--head_lr", type=float, default=1e-3)
    p.add_argument("--wd", type=float, default=0.01)
    p.add_argument("--warmup", type=float, default=0.06)
    p.add_argument("--class_weight", default="sqrt", choices=["none", "sqrt"])
    p.add_argument("--label_smoothing", type=float, default=0.0, help="regularisation for small data / confusable roles")
    p.add_argument("--patience", type=int, default=4)
    p.add_argument("--seed", type=int, default=13)
    p.add_argument("--train_domains", default="CL,IT", help="CL, IT or CL,IT")
    p.add_argument("--data_dir", required=True)
    p.add_argument("--tok", required=True)
    p.add_argument("--ckpt_dir", required=True, help="where Phase-1 checkpoints are (runs/<run>/<file>)")
    p.add_argument("--out_dir", required=True)
    p.add_argument("--hf_repo", default="")
    p.add_argument("--max_steps", type=int, default=-1, help="debug")
    return p.parse_args(argv)


def hf_up(repo, local, remote):
    if not repo:
        return
    from huggingface_hub import HfApi
    for a in range(3):
        try:
            HfApi(token=os.environ.get("HF_TOKEN")).upload_file(path_or_fileobj=local, path_in_repo=remote,
                                                                 repo_id=repo, repo_type="model")
            return
        except Exception as e:
            print("[hf] upload failed:", e, flush=True); time.sleep(15 * (a + 1))


def get_ckpt(args, run, fname):
    local = os.path.join(args.ckpt_dir, "runs", run, fname)
    if not os.path.exists(local) and args.hf_repo:
        from huggingface_hub import hf_hub_download
        hf_hub_download(args.hf_repo, f"runs/{run}/{fname}", repo_type="model", local_dir=args.ckpt_dir,
                        token=os.environ.get("HF_TOKEN"))
    return local


def load_all(data_dir, tok, max_sent_tok):
    LABELS = load_vocab(os.path.join(data_dir, "label_vocab.json"))
    L2I = {l: i for i, l in enumerate(LABELS)}
    split = {}
    for part in ["train", "dev", "test"]:
        docs = []
        for dom in DOMAINS:
            for did, sents, labs in load_docs(os.path.join(data_dir, f"{dom}_{part}.json"), LABELS):
                docs.append(Doc(did, dom, sents, labs if part != "test" else None, tok, L2I, max_sent_tok))
        split[part] = docs
    ids = [d.id for d in split["test"]]
    assert len(ids) == len(set(ids)), "duplicate test document ids across CL and IT"
    return LABELS, L2I, split


def main(argv=None):
    args = parse_args(argv)
    torch.manual_seed(args.seed); random.seed(args.seed); np.random.seed(args.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cuda = dev.type == "cuda"
    amp = (lambda: torch.autocast("cuda", dtype=torch.float16)) if cuda else (lambda: torch.autocast("cpu", enabled=False))
    run_dir = os.path.join(args.out_dir, args.run_name); os.makedirs(run_dir, exist_ok=True)
    logf = open(os.path.join(run_dir, "log.txt"), "a")

    def log(*m):
        s = f"[{time.strftime('%H:%M:%S')}][{args.run_name}] " + " ".join(str(x) for x in m)
        print(s, flush=True); logf.write(s + "\n"); logf.flush()

    # ---------------- data ----------------
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(args.tok)
    t0 = time.time()
    LABELS, L2I, split = load_all(args.data_dir, tok, args.max_sent_tok)
    C = len(LABELS)
    doms = [x.strip() for x in args.train_domains.split(",") if x.strip()]
    train_docs = [d for d in split["train"] if d.domain in doms]
    dev_docs, test_docs = split["dev"], split["test"]
    log(f"labels {C} | train docs {len(train_docs)} ({','.join(doms)}) sentences {sum(d.n for d in train_docs):,} tokens "
        f"{sum(len(d.ids) for d in train_docs):,} | dev docs {len(dev_docs)} sentences {sum(d.n for d in dev_docs):,} | "
        f"test docs {len(test_docs)} sentences {sum(d.n for d in test_docs):,} | prep {time.time() - t0:.0f}s")

    # ---------------- model ----------------
    ck = torch.load(get_ckpt(args, args.base_run, args.base_file), map_location="cpu", weights_only=False)
    lm_cfg = ck["config"]
    if lm_cfg["arch"] == "gpt2":                                # learned positions: never exceed the trained length
        for k in ("win", "infer_win"):
            setattr(args, k, min(getattr(args, k), lm_cfg["max_seq_len"]))
        args.infer_stride = min(args.infer_stride, args.infer_win)
        args.max_sent_tok = min(args.max_sent_tok, lm_cfg["max_seq_len"])
    if args.ctx == "doc":
        args.head = "doclstm"
    a = vars(args)
    net = build_net(lm_cfg, C, a)
    missing, unexpected = net.lm.load_state_dict(ck["model"], strict=False)
    bad = [k for k in missing if not k.startswith("blocks.")] if net.arch == "encdec" else list(missing)
    assert not bad, f"missing backbone weights: {bad[:5]}"
    del ck
    setup_method(net, a)
    net.to(dev)
    total, trainable = count_params(net)
    log(f"backbone {args.base_run} ({lm_cfg['arch']}) | method {args.method}"
        f"{' + KD from ' + args.kd_teacher if args.kd_teacher else ''} | ctx {args.ctx} head {args.head} bidir {args.bidir}"
        f" | params total {total / 1e6:.2f}M trainable {trainable / 1e6:.3f}M ({100 * trainable / total:.2f}%)")

    teacher = None
    if args.kd_teacher:
        tpath = os.path.join(args.out_dir, args.kd_teacher, "best.pt")
        if not os.path.exists(tpath) and args.hf_repo:
            from huggingface_hub import hf_hub_download
            hf_hub_download(args.hf_repo, f"phase3/runs/{args.kd_teacher}/best.pt", repo_type="model",
                            local_dir=os.path.join(args.out_dir, "_teacher"), token=os.environ.get("HF_TOKEN"))
            tpath = os.path.join(args.out_dir, "_teacher", "phase3", "runs", args.kd_teacher, "best.pt")
        tck = torch.load(tpath, map_location="cpu", weights_only=False)
        teacher = build_net(tck["lm_config"], C, tck["args"])
        setup_method(teacher, tck["args"])
        teacher.load_state_dict(tck["state_dict"])
        teacher.to(dev).eval()
        for p in teacher.parameters():
            p.requires_grad = False
        ta = tck["args"]
        if args.ctx == "doc":                                   # document mode: teacher probabilities once per document
            teacher_probs = {d.id: torch.tensor(predict_probs(teacher, d, ta["ctx"], ta["infer_win"], ta["infer_stride"], dev, amp),
                                                dtype=torch.float, device=dev) for d in train_docs}
        tt, _ = count_params(teacher)
        log(f"teacher {args.kd_teacher}: {tt / 1e6:.2f}M params (student {total / 1e6:.2f}M = {100 * total / tt:.0f}% of teacher)")
        del tck

    # ---------------- loss weights ----------------
    cnt = Counter(y for d in train_docs for y in d.y)
    if args.class_weight == "sqrt":
        mx = max(cnt.values())
        w = torch.tensor([math.sqrt(mx / max(1, cnt[c])) for c in range(C)], dtype=torch.float).clamp(1.0, 10.0)
    else:
        w = torch.ones(C)
    w = w.to(dev)
    log("train label counts: " + ", ".join(f"{LABELS[c]}={cnt[c]}" for c in range(C)))

    # ---------------- optimiser ----------------
    head_params = [p for n_, p in net.named_parameters() if p.requires_grad and not n_.startswith("lm.")]
    body_params = [p for n_, p in net.named_parameters() if p.requires_grad and n_.startswith("lm.")]
    body_lr = args.peft_lr if args.method in ("lora", "rslora", "lorma", "ia3") else args.lr
    groups = [{"params": head_params, "lr": args.head_lr}]
    if body_params:
        groups.append({"params": body_params, "lr": body_lr})
    opt = torch.optim.AdamW(groups, weight_decay=args.wd)
    base_lrs = [g["lr"] for g in groups]
    rng = random.Random(args.seed)
    steps_per_epoch = len(train_docs) if args.ctx == "doc" else len(
        batches_by_length(train_items(train_docs, args.ctx, args.win, random.Random(0)), train_docs, args.bs, random.Random(0)))
    total_steps = steps_per_epoch * args.epochs
    warm = max(1, int(args.warmup * total_steps))
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    log(f"steps/epoch {steps_per_epoch} | total {total_steps} | warmup {warm} | lr body {body_lr} head {args.head_lr}")

    def evaluate(docs):
        net.eval()
        pr = {d.id: predict_probs(net, d, args.ctx, args.infer_win, args.infer_stride, dev, amp) for d in docs}
        net.train()
        return pr

    prior = train_prior(train_docs, C)

    def score(docs, probs, tau=0.0):
        truth = {d.id: [LABELS[y] for y in d.y] for d in docs}
        pred = {d.id: [LABELS[int(k)] for k in prior_adjust(probs[d.id], prior, tau).argmax(-1)] for d in docs}
        res = macro_scores(truth, pred, LABELS)
        for dom in DOMAINS:
            ids = [d.id for d in docs if d.domain == dom]
            res[f"f1_{dom}"] = macro_scores({k: truth[k] for k in ids}, {k: pred[k] for k in ids}, LABELS)["f1"] if ids else None
        return res, pred

    # ---------------- training ----------------
    best, best_ep, bad_eps, step, hist = -1.0, -1, 0, 0, []
    t_train = time.time()
    if cuda:
        torch.cuda.reset_peak_memory_stats()
    net.train()
    for ep in range(1, args.epochs + 1):
        tl, tn = 0.0, 0
        if args.ctx == "doc":
            order = [d for d in train_docs if d.n > 0]; rng.shuffle(order)
            for d in order:
                lr_mult = min(1.0, (step + 1) / warm) * (0.5 * (1 + math.cos(math.pi * min(1.0, step / total_steps))))
                for g, b in zip(opt.param_groups, base_lrs):
                    g["lr"] = b * max(lr_mult, 0.02)
                off = rng.randrange(0, args.win)
                nwin = len(range(0, len(d.ids), args.win)) + 1
                g0 = rng.randrange(0, max(1, nwin - args.grad_windows + 1))
                logits = doc_logits(net, d, args.win, off, dev, amp, grad_win=(g0, g0 + args.grad_windows))
                y = torch.tensor(d.y, dtype=torch.long, device=dev)
                loss = F.cross_entropy(logits, y, weight=w, label_smoothing=args.label_smoothing)
                if teacher is not None:
                    T = args.kd_T
                    q = teacher_probs[d.id].clamp(min=1e-8) ** (1.0 / T)
                    q = q / q.sum(-1, keepdim=True)                      # = softmax(teacher_logits / T)
                    kd = F.kl_div(F.log_softmax(logits / T, -1), q, reduction="batchmean") * T * T
                    loss = args.kd_alpha * loss + (1 - args.kd_alpha) * kd
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], 1.0)
                scaler.step(opt); scaler.update()
                tl += loss.item(); tn += 1; step += 1
                if args.max_steps > 0 and step >= args.max_steps:
                    break
        for batch in ([] if args.ctx == "doc" else
                      batches_by_length(train_items(train_docs, args.ctx, args.win, rng), train_docs, args.bs, rng)):
            lr_mult = min(1.0, (step + 1) / warm) * (0.5 * (1 + math.cos(math.pi * min(1.0, step / total_steps))))
            for g, b in zip(opt.param_groups, base_lrs):
                g["lr"] = b * max(lr_mult, 0.02)
            x, seg, S, lens, slots = collate(batch, train_docs, dev)
            yy = torch.full((len(batch), S), -100, dtype=torch.long)
            ww = torch.zeros((len(batch), S))
            for r, ((di, _, _, _), sl) in enumerate(zip(batch, slots)):
                for k, (s, share, _) in enumerate(sl):
                    yy[r, k] = train_docs[di].y[s]; ww[r, k] = share
            yy, ww = yy.to(dev), ww.to(dev)
            with amp():
                logits = net(x, seg, S, lens)
            m = yy >= 0
            ce = F.cross_entropy(logits[m], yy[m], weight=w, reduction="none", label_smoothing=args.label_smoothing)
            wm = ww[m]
            loss = (ce * wm).sum() / wm.sum().clamp(min=1e-6)
            if teacher is not None:
                with torch.no_grad(), amp():
                    tlog = teacher(x, seg, S, lens).float()
                T = args.kd_T
                kl = F.kl_div(F.log_softmax(logits[m] / T, -1), F.softmax(tlog[m] / T, -1), reduction="none").sum(-1)
                kd = (kl * wm).sum() / wm.sum().clamp(min=1e-6) * T * T
                loss = args.kd_alpha * loss + (1 - args.kd_alpha) * kd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], 1.0)
            scaler.step(opt); scaler.update()
            tl += loss.item(); tn += 1; step += 1
            if args.max_steps > 0 and step >= args.max_steps:
                break
        probs = evaluate(dev_docs)
        res = max((score(dev_docs, probs, tau)[0] for tau in TAUS), key=lambda r: r["f1"])   # best calibration
        hist.append({"epoch": ep, "loss": tl / max(1, tn), "dev_f1": res["f1"], "dev_f1_CL": res["f1_CL"],
                     "dev_f1_IT": res["f1_IT"], "dev_acc": res["acc"]})
        log(f"epoch {ep:2d} loss {tl / max(1, tn):.4f} | dev official macro-F1 {res['f1']:.4f} (CL {res['f1_CL']:.4f} "
            f"IT {res['f1_IT']:.4f}) acc {res['acc']:.4f} | {(time.time() - t_train) / 60:.1f} min")
        if res["f1"] > best:
            best, best_ep, bad_eps = res["f1"], ep, 0
            torch.save({"state_dict": net.state_dict(), "lm_config": lm_cfg, "args": a, "labels": LABELS,
                        "epoch": ep, "dev_f1": best}, os.path.join(run_dir, "best.pt"))
        else:
            bad_eps += 1
            if bad_eps >= args.patience:
                log(f"early stop (no improvement for {args.patience} epochs)"); break
        if args.max_steps > 0 and step >= args.max_steps:
            break
    train_min = (time.time() - t_train) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 if cuda else 0.0

    # ---------------- final: best checkpoint -> dev scores + test predictions ----------------
    net.load_state_dict(torch.load(os.path.join(run_dir, "best.pt"), map_location=dev, weights_only=False)["state_dict"])
    t_inf = time.time()
    dev_probs = evaluate(dev_docs)
    infer_dev_s = time.time() - t_inf
    res_raw, _ = score(dev_docs, dev_probs, 0.0)
    tau = max(TAUS, key=lambda t_: (score(dev_docs, dev_probs, t_)[0]["f1"], -t_))     # ties -> smaller tau
    res, dev_pred = score(dev_docs, dev_probs, tau)
    conf = np.zeros((C, C), dtype=int)                     # confusion matrix on dev: rows = gold, cols = predicted
    for d in dev_docs:
        for g, pl in zip(d.y, dev_pred[d.id]):
            conf[g, L2I[pl]] += 1
    log(f"DEV best epoch {best_ep}: official macro-F1 {res['f1']:.4f} (P {res['p']:.4f} R {res['r']:.4f}) with prior "
        f"calibration tau={tau} | without calibration {res_raw['f1']:.4f} | CL {res['f1_CL']:.4f} IT {res['f1_IT']:.4f} "
        f"| acc {res['acc']:.4f}")
    log("DEV per label F1: " + " ".join(f"{l}={res['per_label'][l][2]:.3f}" for l in LABELS))

    t_inf = time.time()
    test_probs = evaluate(test_docs)
    infer_test_s = time.time() - t_inf
    sub = {d.id: [LABELS[int(k)] for k in prior_adjust(test_probs[d.id], prior, tau).argmax(-1)] for d in test_docs}
    # ---- validate the submission before writing it ----
    assert set(sub) == {d.id for d in test_docs}, "test ids mismatch"
    for d in test_docs:
        assert len(sub[d.id]) == d.n == len(d.sents), f"{d.id}: {len(sub[d.id])} predictions for {d.n} sentences"
        assert all(l in L2I for l in sub[d.id])
    pj = os.path.join(run_dir, "prediction.json")
    json.dump(sub, open(pj, "w"))
    zp = os.path.join(run_dir, "submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    json.dump(dev_pred, open(os.path.join(run_dir, "dev_prediction.json"), "w"))
    np.savez_compressed(os.path.join(run_dir, "probs.npz"),
                        **{f"dev__{k}": v.astype(np.float32) for k, v in dev_probs.items()},
                        **{f"test__{k}": v.astype(np.float32) for k, v in test_probs.items()})
    summary = {"run": args.run_name, "base_run": args.base_run, "arch": lm_cfg["arch"], "method": args.method,
               "kd_teacher": args.kd_teacher or None, "bidir": args.bidir, "ctx": args.ctx, "head": args.head,
               "train_domains": args.train_domains,
               "params_total": total, "params_trainable": trainable, "best_epoch": best_ep,
               "dev_f1": res["f1"], "dev_p": res["p"], "dev_r": res["r"], "dev_f1_CL": res["f1_CL"],
               "dev_f1_IT": res["f1_IT"], "dev_f1_raw": res_raw["f1"], "tau": tau, "dev_f1_present": res["f1_present"],
               "dev_acc": res["acc"], "train_prior": prior.tolist(),
               "dev_per_label": res["per_label"], "dev_confusion": conf.tolist(), "labels": LABELS,
               "train_minutes": round(train_min, 2), "peak_gpu_gb": round(peak_gb, 2),
               "infer_dev_seconds": round(infer_dev_s, 1), "infer_test_seconds": round(infer_test_s, 1),
               "test_sentences": sum(d.n for d in test_docs), "history": hist, "args": a}
    json.dump(summary, open(os.path.join(run_dir, "summary.json"), "w"), indent=1)
    log(f"TEST: {summary['test_sentences']} sentences in {len(sub)} docs -> {zp}")
    for fn in ["summary.json", "prediction.json", "submission.zip", "dev_prediction.json", "log.txt", "probs.npz", "best.pt"]:
        if os.path.exists(os.path.join(run_dir, fn)):
            hf_up(args.hf_repo, os.path.join(run_dir, fn), f"phase3/runs/{args.run_name}/{fn}")
    log("DONE")


if __name__ == "__main__":
    main()


Writing /kaggle/working/p3rr.py


In [5]:
%%writefile /kaggle/working/p3ens.py
"""CS-781 Phase 3: ensembles of our fine-tuned RR models (all built on our own Phase-1 models).

Each run saved its sentence probabilities for dev and test (probs.npz; recomputed from best.pt if missing).
An ensemble = plain average of the members' probabilities, then argmax per sentence.
  ens_top3 / ens_top5 / ens_top8 : the K runs with the best dev macro-F1 (no search -> least fitted to dev)
  ens_greedy                     : greedy forward selection on dev
"""
import argparse, json, os, sys, time, zipfile
import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import p3rr as R


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--data_dir", required=True)
    p.add_argument("--tok", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--hf_repo", default="")
    p.add_argument("--min_dev", type=float, default=0.20)
    p.add_argument("--ks", default="3,5,8")
    p.add_argument("--max_members", type=int, default=6)
    p.add_argument("--min_gain", type=float, default=0.002)
    p.add_argument("--seq", type=int, default=0, help="1 = also write sequence-decoded (Viterbi) ensembles (hurt on test)")
    return p.parse_args(argv)


def hf_get(repo, remote, local_dir):
    if not repo:
        return None
    from huggingface_hub import hf_hub_download
    try:
        return hf_hub_download(repo, remote, repo_type="model", local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def main(argv=None):
    args = parse_args(argv)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = (lambda: torch.autocast("cuda", dtype=torch.float16)) if dev.type == "cuda" else (lambda: torch.autocast("cpu", enabled=False))
    t0 = time.time()

    def log(*m):
        print(f"[{time.strftime('%H:%M:%S')}][ensemble] " + " ".join(str(x) for x in m), flush=True)

    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(args.tok)
    LABELS, L2I, split = R.load_all(args.data_dir, tok, 256)
    dev_docs, test_docs = split["dev"], split["test"]
    truth = {d.id: [LABELS[y] for y in d.y] for d in dev_docs}
    prior = R.train_prior(split["train"], len(LABELS))

    # ---------------- which runs exist (local + Hugging Face) ----------------
    names = set(x for x in os.listdir(args.out_dir) if os.path.isdir(os.path.join(args.out_dir, x)))
    if args.hf_repo:
        from huggingface_hub import HfApi
        for f in HfApi(token=os.environ.get("HF_TOKEN")).list_repo_files(args.hf_repo):
            if f.startswith("phase3/runs/") and f.endswith("/summary.json"):
                names.add(f.split("/")[2])
    summ = {}
    for n in sorted(names):
        if n.startswith("ens_") or n.startswith("_"):
            continue
        p = os.path.join(args.out_dir, n, "summary.json")
        if not os.path.exists(p):
            q = hf_get(args.hf_repo, f"phase3/runs/{n}/summary.json", os.path.join(args.out_dir, "_hub"))
            if q is None:
                continue
            os.makedirs(os.path.dirname(p), exist_ok=True)
            import shutil; shutil.copy(q, p)
        summ[n] = json.load(open(p))
    cands = sorted([n for n, s in summ.items() if s.get("train_domains", "CL,IT") == "CL,IT" and s["dev_f1"] >= args.min_dev],
                   key=lambda n: -summ[n]["dev_f1"])
    log(f"{len(cands)} candidate runs:", ", ".join(f"{n}={summ[n]['dev_f1']:.4f}" for n in cands))
    if len(cands) < 2:
        log("fewer than 2 candidates - nothing to ensemble"); return

    # ---------------- probabilities of every candidate ----------------
    def get_probs(run):
        path = os.path.join(args.out_dir, run, "probs.npz")
        if not os.path.exists(path):
            q = hf_get(args.hf_repo, f"phase3/runs/{run}/probs.npz", os.path.join(args.out_dir, "_hub"))
            if q is not None:
                import shutil; shutil.copy(q, path)
        if not os.path.exists(path):                       # recompute from the checkpoint
            bp = os.path.join(args.out_dir, run, "best.pt")
            if not os.path.exists(bp):
                bp = hf_get(args.hf_repo, f"phase3/runs/{run}/best.pt", os.path.join(args.out_dir, "_hub"))
            ck = torch.load(bp, map_location="cpu", weights_only=False)
            a = ck["args"]
            net = R.build_net(ck["lm_config"], len(LABELS), a)
            R.setup_method(net, a)
            net.load_state_dict(ck["state_dict"])
            net.to(dev).eval()
            dp = {d.id: R.predict_probs(net, d, a["ctx"], a["infer_win"], a["infer_stride"], dev, amp) for d in dev_docs}
            tp = {d.id: R.predict_probs(net, d, a["ctx"], a["infer_win"], a["infer_stride"], dev, amp) for d in test_docs}
            np.savez_compressed(path, **{f"dev__{k}": v for k, v in dp.items()}, **{f"test__{k}": v for k, v in tp.items()})
            del net, ck
            log(f"computed probabilities for {run}")
        z = np.load(path)
        return ({d.id: z[f"dev__{d.id}"] for d in dev_docs}, {d.id: z[f"test__{d.id}"] for d in test_docs})

    P = {n: get_probs(n) for n in cands}

    def mean(dicts):
        return {k: sum(d[k] for d in dicts) / len(dicts) for k in dicts[0]}

    def argmax(probs, tau=0.0):
        return {k: [LABELS[int(i)] for i in R.prior_adjust(v, prior, tau).argmax(-1)] for k, v in probs.items()}

    def best_tau(members):
        dp = mean([P[m][0] for m in members])
        return max(R.TAUS, key=lambda t_: (R.macro_scores(truth, argmax(dp, t_), LABELS)["f1"], -t_))

    def dev_score(members):
        return R.macro_scores(truth, argmax(mean([P[m][0] for m in members]), best_tau(members)), LABELS)

    def write(name, members, info):
        pred = argmax(mean([P[m][1] for m in members]), info["tau"])
        for d in test_docs:
            assert len(pred[d.id]) == d.n and all(l in L2I for l in pred[d.id])
        rd = os.path.join(args.out_dir, name); os.makedirs(rd, exist_ok=True)
        json.dump(pred, open(os.path.join(rd, "prediction.json"), "w"))
        with zipfile.ZipFile(os.path.join(rd, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(os.path.join(rd, "prediction.json"), arcname="prediction.json")
        json.dump({"members": members, **info}, open(os.path.join(rd, "summary.json"), "w"), indent=1)
        log(f"{name}: {len(members)} models {members} | dev official macro-F1 {info['dev_f1']:.4f} (tau {info['tau']}) "
            f"(CL {info['dev_f1_CL']:.4f} IT {info['dev_f1_IT']:.4f}) -> {rd}/submission.zip")
        return name

    def info_of(members):
        tau = best_tau(members)
        r = R.macro_scores(truth, argmax(mean([P[m][0] for m in members]), tau), LABELS)
        out = {"tau": tau, "dev_f1": r["f1"], "dev_p": r["p"], "dev_r": r["r"], "dev_acc": r["acc"], "dev_per_label": r["per_label"]}
        for dom in R.DOMAINS:
            ids = [d.id for d in dev_docs if d.domain == dom]
            pr = argmax(mean([{k: P[m][0][k] for k in ids} for m in members]), tau)
            out[f"dev_f1_{dom}"] = R.macro_scores({k: truth[k] for k in ids}, pr, LABELS)["f1"]
        return out

    outs = []
    for K in [int(k) for k in args.ks.split(",") if k.strip()]:
        if K <= len(cands):
            outs.append(write(f"ens_top{K}", cands[:K], info_of(cands[:K])))
    # greedy forward selection on dev
    chosen = [cands[0]]; cur = dev_score(chosen)["f1"]
    while len(chosen) < args.max_members:
        best_n, best_f = None, cur
        for n in cands:
            if n not in chosen:
                f = dev_score(chosen + [n])["f1"]
                if f > best_f:
                    best_n, best_f = n, f
        if best_n is None or best_f - cur < args.min_gain:
            break
        chosen.append(best_n); cur = best_f
    if len(chosen) > 1:
        outs.append(write("ens_greedy", chosen, info_of(chosen)))
    # ---------------- sequence decoding (Viterbi with role-transition probabilities learned from TRAIN labels) ----------------
    # Rhetorical roles come in blocks; instead of labelling each sentence alone we pick the most likely label SEQUENCE:
    #   score(y) = sum_i log q_i(y_i) + lam * [log S(y_0) + sum_i log T(y_{i-1}, y_i)]
    # q = calibrated ensemble probabilities (tau), T/S = transition/start probabilities counted on the training documents.
    C = len(LABELS)
    trans = np.ones((C, C)); start = np.ones(C)
    for d in split["train"]:
        if d.n:
            start[d.y[0]] += 1
        for y0, y1 in zip(d.y[:-1], d.y[1:]):
            trans[y0, y1] += 1
    logT = np.log(trans / trans.sum(1, keepdims=True)); logS = np.log(start / start.sum())
    LAMS = (0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0)

    def viterbi(lp, lam):
        n = lp.shape[0]
        score = lp[0] + lam * logS
        back = np.zeros((n, C), dtype=np.int64)
        for i in range(1, n):
            cand = score[:, None] + lam * logT
            back[i] = cand.argmax(0); score = cand.max(0) + lp[i]
        ys = [int(score.argmax())]
        for i in range(n - 1, 0, -1):
            ys.append(int(back[i, ys[-1]]))
        return ys[::-1]

    def seq_decode(probs, tau, lam):
        out = {}
        for k, v in probs.items():
            q = R.prior_adjust(v, prior, tau)
            ys = q.argmax(-1) if lam == 0 or len(q) == 0 else viterbi(np.log(q + 1e-12), lam)
            out[k] = [LABELS[int(i)] for i in ys]
        return out

    def write_seq(name, members):
        dp = mean([P[m][0] for m in members]); tp = mean([P[m][1] for m in members])
        best = max(((tau, lam) for tau in R.TAUS for lam in LAMS),
                   key=lambda tl: (R.macro_scores(truth, seq_decode(dp, *tl), LABELS)["f1"], -tl[1], -tl[0]))
        r = R.macro_scores(truth, seq_decode(dp, *best), LABELS)
        r0 = R.macro_scores(truth, seq_decode(dp, best[0], 0.0), LABELS)
        info = {"tau": best[0], "lam": best[1], "dev_f1": r["f1"], "dev_f1_no_seq": r0["f1"], "dev_p": r["p"], "dev_r": r["r"],
                "dev_acc": r["acc"], "dev_per_label": r["per_label"], "transitions_from": "train labels"}
        for dom in R.DOMAINS:
            ids = [d.id for d in dev_docs if d.domain == dom]
            pr = seq_decode({k: dp[k] for k in ids}, *best)
            info[f"dev_f1_{dom}"] = R.macro_scores({k: truth[k] for k in ids}, pr, LABELS)["f1"]
        pred = seq_decode(tp, *best)
        for d in test_docs:
            assert len(pred[d.id]) == d.n and all(l in L2I for l in pred[d.id])
        rd = os.path.join(args.out_dir, name); os.makedirs(rd, exist_ok=True)
        json.dump(pred, open(os.path.join(rd, "prediction.json"), "w"))
        with zipfile.ZipFile(os.path.join(rd, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(os.path.join(rd, "prediction.json"), arcname="prediction.json")
        json.dump({"members": members, **info}, open(os.path.join(rd, "summary.json"), "w"), indent=1)
        log(f"{name}: {len(members)} models | dev official macro-F1 {r['f1']:.4f} with sequence decoding (tau {best[0]}, "
            f"lambda {best[1]}) vs {r0['f1']:.4f} without | CL {info['dev_f1_CL']:.4f} IT {info['dev_f1_IT']:.4f} -> {rd}/submission.zip")
        return name

    seq_sets = [(f"ens_top{K}_seq", cands[:K]) for K in [int(k) for k in args.ks.split(",") if k.strip()] if K <= len(cands)]
    if len(chosen) > 1:
        seq_sets.append(("ens_greedy_seq", chosen))
    seq_sets.append(("best1_seq", cands[:1]))
    for name, members in (seq_sets if args.seq else []):
        outs.append(write_seq(name, members))
    if args.hf_repo:
        for o in outs:
            for fn in ["summary.json", "prediction.json", "submission.zip"]:
                R.hf_up(args.hf_repo, os.path.join(args.out_dir, o, fn), f"phase3/runs/{o}/{fn}")
    log(f"DONE in {(time.time() - t0) / 60:.1f} min")


if __name__ == "__main__":
    main()


Writing /kaggle/working/p3ens.py


In [6]:
# ===================== P3 CELL 3 - Look at the data (CPU, ~1 min) =====================
sys.path.insert(0, WORK)
import importlib, p2text, p3rr; importlib.reload(p2text); importlib.reload(p3rr)
import numpy as np
LABELS, L2I, SPLIT = p3rr.load_all(RAW, tok, 256)
print("labels:", LABELS)
stats = {}
for part in ["train", "dev", "test"]:
    for dom in p3rr.DOMAINS:
        ds = [d for d in SPLIT[part] if d.domain == dom]
        ns = [d.n for d in ds]; nt = [t for d in ds for t in d.sent_tok]
        line = (f"{dom} {part:5s}: {len(ds):3d} docs | sentences {sum(ns):6,} (per doc mean {np.mean(ns):6.1f} max {max(ns):5d})"
                f" | tokens/sentence mean {np.mean(nt):5.1f} max {max(nt):4d} | tokens/doc mean {np.mean([len(d.ids) for d in ds]):7,.0f}")
        print(line)
        st = {"docs": len(ds), "sentences": int(sum(ns)), "tok_per_sent": float(np.mean(nt))}
        if part != "test":
            c = collections.Counter(LABELS[y] for d in ds for y in d.y)
            st["labels"] = {l: c.get(l, 0) for l in LABELS}
        stats[f"{dom}_{part}"] = st
print("\nlabel counts (sentences):")
print(f"{'label':24s}" + "".join(f"{k:>10s}" for k in ["CL_train", "IT_train", "CL_dev", "IT_dev"]))
for l in LABELS:
    print(f"{l:24s}" + "".join(f"{stats[k]['labels'][l]:10d}" for k in ["CL_train", "IT_train", "CL_dev", "IT_dev"]))
empty = sum(1 for part in SPLIT for d in SPLIT[part] for s in d.sents if not s.strip())
print("\nempty sentences (all splits):", empty, "| test doc ids:", [d.id for d in SPLIT["test"]])
os.makedirs(f"{P3}/data", exist_ok=True)
json.dump(stats, open(f"{P3}/data/stats.json", "w"), indent=1)
api.upload_file(path_or_fileobj=f"{P3}/data/stats.json", path_in_repo="phase3/data/stats.json", repo_id=CKPT_REPO, repo_type="model")
print("uploaded data statistics to", f"{CKPT_REPO}/phase3/data/stats.json")


No files have been modified since last commit. Skipping to prevent empty commit.


labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
CL train:  40 docs | sentences 10,593 (per doc mean  264.8 max  1816) | tokens/sentence mean  39.1 max  256 | tokens/doc mean  10,356
IT train:  40 docs | sentences  6,252 (per doc mean  156.3 max   442) | tokens/sentence mean  42.2 max  256 | tokens/doc mean   6,603
CL dev  :   5 docs | sentences  1,396 (per doc mean  279.2 max   848) | tokens/sentence mean  40.6 max  256 | tokens/doc mean  11,323
IT dev  :   5 docs | sentences    746 (per doc mean  149.2 max   382) | tokens/sentence mean  39.3 max  256 | tokens/doc mean   5,857
CL test :   5 docs | sentences  1,339 (per doc mean  267.8 max   942) | tokens/sentence mean  38.7 max  256 | tokens/doc mean  10,375
IT test :   5 docs | sentences    858 (per doc mean  171.6 max   420) | tokens/sentence mea

In [7]:
# ===================== P3 CELL 7 - Run the experiments on both GPUs (queue with dependencies, resumable) =====================
import threading, subprocess
CK = "/tmp/cs781/ckpt"                    # Phase-1 checkpoints (downloaded once, before the workers start)
OUT = f"{WORK}/p3runs"; os.makedirs(OUT, exist_ok=True)
for run, fn in [("dapt_llama", "best_model.pt"), ("explore_gpt2", "final_model.pt"),
                ("explore_encdec", "final_model.pt"), ("explore_shared", "final_model.pt")]:
    hf_hub_download(CKPT_REPO, f"runs/{run}/{fn}", repo_type="model", local_dir=CK, token=HF_TOKEN)
print("Phase-1 checkpoints ready")

COMMON = dict(epochs=15, win=512, infer_win=512, infer_stride=256, bs=16, lr=5e-5, peft_lr=1e-3, head_lr=1e-3,
              rank=16, alpha=32, patience=4)
LLAMA  = dict(base_run="dapt_llama", base_file="best_model.pt")
GPT2   = dict(base_run="explore_gpt2", base_file="final_model.pt", bidir=0)        # GPT-2: left-to-right (Phase 2 lesson)
ENCDEC = dict(base_run="explore_encdec", base_file="final_model.pt")
SHARED = dict(base_run="explore_shared", base_file="final_model.pt")
DOC = dict(ctx="doc", epochs=20, patience=5)                                          # WHOLE-DOCUMENT model
ALL7 = "q,k,v,o,gate,up,down"                                                      # attention + feed-forward (MLP) layers
R4 = dict(ctx="doc", epochs=20, patience=5, targets=ALL7)
RUNS4 = [  # ---- round 4 (runs FIRST): adapters also in the feed-forward layers, label smoothing, 2nd seed ----
    ("p4_llama_lorma_ffn",      {**LLAMA, **R4, "method": "lorma"}),
    ("p4_llama_lora_ffn",       {**LLAMA, **R4, "method": "lora"}),
    ("p4_llama_lorma_ffn_ls",   {**LLAMA, **R4, "method": "lorma", "label_smoothing": 0.1}),
    ("p4_llama_lorma_ffn_s14",  {**LLAMA, **R4, "method": "lorma", "seed": 14}),
    ("p4_gpt2_lorma_ffn",       {**GPT2,  **R4, "method": "lorma", "targets": "q,k,v,o,up,down"}),
    ("p4_llama_lorma_ffn_lr3e4",{**LLAMA, **R4, "method": "lorma", "peft_lr": 3e-4}),
]
LONG = dict(ctx="doc", epochs=35, patience=8)                                        # round 3: best runs were still improving at 20
RUNS = RUNS4 + [   # ---- round 3: longer training of the best document-level models + a 2nd seed ----
    ("p3d_llama_lorma_e35",     {**LLAMA, **LONG, "method": "lorma"}),
    ("p3d_llama_lora_e35",      {**LLAMA, **LONG, "method": "lora"}),
    ("p3d_llama_lorma_e35_s14", {**LLAMA, **LONG, "method": "lorma", "seed": 14}),
    ("p3d_gpt2_lorma_e35",      {**GPT2,  **LONG, "method": "lorma"}),
]
RUNS += [   # ---- round 2: document-level model = sentence vectors + position -> BiLSTM over ALL sentences ----
    ("p3d_llama_lorma",      {**LLAMA,  **DOC, "method": "lorma"}),                       # LoRMA (compulsory), KD teacher
    ("p3d_llama_lora",       {**LLAMA,  **DOC, "method": "lora"}),
    ("p3d_shared_kd",        {**SHARED, **DOC, "method": "full", "lr": 1e-4, "kd_teacher": "p3d_llama_lorma"}),  # KD
    ("p3d_gpt2_lorma",       {**GPT2,   **DOC, "method": "lorma"}),
    ("p3d_encdec_lorma",     {**ENCDEC, **DOC, "method": "lorma"}),
    ("p3d_llama_full",       {**LLAMA,  **DOC, "method": "full"}),
    ("p3d_llama_ia3",        {**LLAMA,  **DOC, "method": "ia3", "peft_lr": 5e-3}),
    ("p3d_llama_rslora",     {**LLAMA,  **DOC, "method": "rslora", "rank": 64, "alpha": 16}),
    ("p3d_shared_full",      {**SHARED, **DOC, "method": "full", "lr": 1e-4}),                # same student WITHOUT KD
]
RUNS += [   # ---- round 1: techniques on llama (context: 512-token window + BiLSTM over the window's sentences) ----
    ("p3_llama_lorma",       {**LLAMA,  "method": "lorma"}),                              # LoRMA (compulsory), KD teacher
    ("p3_llama_lora",        {**LLAMA,  "method": "lora"}),                               # LoRA
    ("p3_llama_rslora",      {**LLAMA,  "method": "rslora", "rank": 64, "alpha": 16}),     # rank-stabilised LoRA
    ("p3_llama_ia3",         {**LLAMA,  "method": "ia3", "peft_lr": 5e-3}),                # IA3
    ("p3_shared_kd",         {**SHARED, "method": "full", "lr": 1e-4, "kd_teacher": "p3_llama_lorma"}),  # knowledge distillation
           # ---- all four Phase-1 models ----
    ("p3_gpt2_lorma",        {**GPT2,   "method": "lorma"}),
    ("p3_encdec_lorma",      {**ENCDEC, "method": "lorma"}),
    ("p3_shared_full",       {**SHARED, "method": "full", "lr": 1e-4}),                    # same student WITHOUT KD
           # ---- context study (sentence-only vs token window vs window + sentence BiLSTM) ----
    ("p3_llama_lorma_sent",  {**LLAMA,  "method": "lorma", "ctx": "sent", "head": "linear"}),
    ("p3_llama_lorma_win",   {**LLAMA,  "method": "lorma", "ctx": "win", "head": "linear"}),
           # ---- full-model adaptation + transfer ----
    ("p3_llama_full",        {**LLAMA,  "method": "full"}),
    ("p3_llama_head",        {**LLAMA,  "method": "head"}),                               # frozen Phase-1 model, probe only
    ("p3_gpt2_full",         {**GPT2,   "method": "full"}),
    ("p3_encdec_lora",       {**ENCDEC, "method": "lora"}),
           # ---- domain study (one domain only) ----
    ("p3_llama_lorma_CLonly", {**LLAMA, "method": "lorma", "train_domains": "CL"}),
    ("p3_llama_lorma_ITonly", {**LLAMA, "method": "lorma", "train_domains": "IT"}),
]
ONLY = None          # e.g. ["p3_llama_lorma", "p3_shared_kd"] to run a subset

def p3_cmd(name, cfg, out_dir, hf_repo):
    cmd = [sys.executable, f"{WORK}/p3rr.py", "--run_name", name, "--data_dir", RAW, "--tok", TOK_PATH,
           "--ckpt_dir", CK, "--out_dir", out_dir, "--hf_repo", hf_repo]
    for k, v in {**COMMON, **cfg}.items():
        cmd += [f"--{k}", str(v)]
    return cmd

# ---- safety test: a few training steps + full dev/test prediction + zip on the REAL data, nothing uploaded ----
t_s = time.time()
smoke = p3_cmd("_smoke", {**SHARED, "method": "lora", "epochs": 1, "max_steps": 3}, f"{P3}/smoke", "")
r = subprocess.run(smoke, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"), capture_output=True, text=True)
print("\n".join([l for l in (r.stdout + r.stderr).splitlines() if "][_smoke]" in l][-6:]))
if r.returncode != 0 or not os.path.exists(f"{P3}/smoke/_smoke/submission.zip"):
    print("\n".join((r.stdout + r.stderr).splitlines()[-30:]))
    raise RuntimeError("SAFETY TEST FAILED - nothing was trained. Send this output to Claude.")
print(f"safety test passed in {time.time() - t_s:.0f}s")
if torch.cuda.device_count() == 0:
    print("\nCPU DRY RUN PASSED on the real data. Now switch Accelerator to GPU T4 x2 and run again.")
    RUNS = []
else:
    print("starting the experiments\n")

hub_files = set(api.list_repo_files(CKPT_REPO))
done, failed, running, lock = set(), set(), set(), threading.Lock()
pending = [(n, c) for n, c in RUNS if ONLY is None or n in ONLY]
for n, c in list(pending):
    if f"phase3/runs/{n}/summary.json" in hub_files:          # finished in an earlier session -> skip
        os.makedirs(f"{OUT}/{n}", exist_ok=True)
        for fn in ["summary.json", "submission.zip", "prediction.json"]:
            if f"phase3/runs/{n}/{fn}" in hub_files:
                p = hf_hub_download(CKPT_REPO, f"phase3/runs/{n}/{fn}", repo_type="model", local_dir=f"{P3}/hub", token=HF_TOKEN)
                shutil.copy(p, f"{OUT}/{n}/{fn}")
        done.add(n); pending.remove((n, c)); print("already done:", n)

def next_job():
    with lock:
        for i, (n, c) in enumerate(pending):
            t = c.get("kd_teacher")
            if t and t not in done and f"phase3/runs/{t}/best.pt" in hub_files:
                done.add(t)                                   # teacher finished in an earlier session (on HF)
            if t and t not in done and t not in running and all(t != m for m, _ in pending):
                failed.add(t)                                 # teacher is not available and not scheduled
            if t and t in failed:
                pending.pop(i); failed.add(n); print(f"SKIP {n}: teacher {t} failed / not available", flush=True); return "AGAIN"
            if not t or t in done:
                running.add(n); return pending.pop(i)
        return "WAIT" if pending else None

def worker(gpu):
    while True:
        job = next_job()
        if job is None: return
        if job in ("WAIT", "AGAIN"):
            time.sleep(30 if job == "WAIT" else 0); continue
        name, cfg = job
        with open(f"{OUT}/{name}.out", "w") as f:
            r = subprocess.run(p3_cmd(name, cfg, OUT, CKPT_REPO), env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)),
                               stdout=f, stderr=subprocess.STDOUT)
        ok_ = r.returncode == 0 and os.path.exists(f"{OUT}/{name}/summary.json")
        with lock:
            (done if ok_ else failed).add(name); running.discard(name)
            print(f"{'FINISHED' if ok_ else 'FAILED  '} {name} on GPU {gpu}", flush=True)
            if not ok_:
                print(f"===== last lines of {name}.out =====\n" + "\n".join(open(f"{OUT}/{name}.out").read().splitlines()[-30:]), flush=True)

def status_line(name):
    p = f"{OUT}/{name}.out"
    if not os.path.exists(p): return "(waiting)"
    keep = [l for l in open(p).read().splitlines() if "epoch" in l or "DEV" in l or "Error" in l or "DONE" in l]
    return keep[-1][:170] if keep else "(starting)"

threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads: t.start()
t0 = time.time()
while any(t.is_alive() for t in threads):
    for t in threads: t.join(timeout=300 / len(threads))
    print(f"--- {(time.time() - t0) / 60:.0f} min | done {sorted(done)} | failed {sorted(failed)}", flush=True)
    for n, _ in RUNS:
        if n not in done and n not in failed and (ONLY is None or n in ONLY): print(f"   {n:22s} {status_line(n)}", flush=True)
print("ALL FINISHED. done:", sorted(done), "| failed:", sorted(failed))


runs/dapt_llama/best_model.pt: reconstructing file:   0%|          |  0.00B /  431MB            

runs/dapt_llama/best_model.pt: downloading bytes:           |  0.00B            

runs/explore_gpt2/final_model.pt: reconstructing file:   0%|          |  0.00B /  444MB            

runs/explore_gpt2/final_model.pt: downloading bytes:           |  0.00B            

runs/explore_encdec/final_model.pt: reconstructing file:   0%|          |  0.00B /  469MB            

runs/explore_encdec/final_model.pt: downloading bytes:           |  0.00B            

... (144 outputs left out)

phase3/runs/p3_llama_lorma_ITonly/submis(…): reconstructing file:   0%|          |  0.00B / 1.65kB            

phase3/runs/p3_llama_lorma_ITonly/submis(…): downloading bytes:           |  0.00B            

prediction.json:   0%|          | 0.00/41.6k [00:00<?, ?B/s]

already done: p3_llama_lorma_ITonly
--- 5 min | done ['p3_encdec_lora', 'p3_encdec_lorma', 'p3_gpt2_full', 'p3_gpt2_lorma', 'p3_llama_full', 'p3_llama_head', 'p3_llama_ia3', 'p3_llama_lora', 'p3_llama_lorma', 'p3_llama_lorma_CLonly', 'p3_llama_lorma_ITonly', 'p3_llama_lorma_sent', 'p3_llama_lorma_win', 'p3_llama_rslora', 'p3_shared_full', 'p3_shared_kd', 'p3d_encdec_lorma', 'p3d_gpt2_lorma', 'p3d_gpt2_lorma_e35', 'p3d_llama_full', 'p3d_llama_ia3', 'p3d_llama_lora', 'p3d_llama_lora_e35', 'p3d_llama_lorma', 'p3d_llama_lorma_e35', 'p3d_llama_lorma_e35_s14', 'p3d_llama_rslora', 'p3d_shared_full', 'p3d_shared_kd'] | failed []
   p4_llama_lorma_ffn     [04:37:18][p4_llama_lorma_ffn] epoch  6 loss 1.2795 | dev official macro-F1 0.3444 (CL 0.2739 IT 0.3770) acc 0.4897 | 4.7 min
   p4_llama_lora_ffn      [04:36:52][p4_llama_lora_ffn] epoch  5 loss 1.1479 | dev official macro-F1 0.3602 (CL 0.2934 IT 0.3868) acc 0.4729 | 4.3 min
   p4_llama_lorma_ffn_ls  (waiting)
   p4_llama_lorma_ffn_s14 (waiti

In [8]:
# ===================== P3 CELL 8 - Results table + submission zips =====================
ZIPS = f"{WORK}/zips"; os.makedirs(ZIPS, exist_ok=True)
rows = []
for n, _ in RUNS if RUNS else []:
    p = f"{OUT}/{n}/summary.json"
    if not os.path.exists(p): continue
    rows.append(json.load(open(p)))
    if os.path.exists(f"{OUT}/{n}/submission.zip"):
        shutil.copy(f"{OUT}/{n}/submission.zip", f"{ZIPS}/{n}.zip")
rows.sort(key=lambda s: -s["dev_f1"])
print(f"{'run':24s} {'model':7s} {'method':7s} {'ctx':10s} {'params':>8s} {'trainable':>10s} {'devF1':>7s} {'CL':>6s} {'IT':>6s} "
      f"{'raw':>6s} {'tau':>4s} {'acc':>6s} {'ep':>3s} {'train':>6s} {'GPU GB':>6s} {'test s':>6s}")
for s in rows:
    m = s["method"] + ("+KD" if s.get("kd_teacher") else "")
    c = s["ctx"] + ("+lstm" if s["head"] == "bilstm" else "") + ("" if s.get("train_domains", "CL,IT") == "CL,IT" else f"/{s['train_domains']}")
    print(f"{s['run']:24s} {s['arch']:7s} {m:7s} {c:10s} {s['params_total'] / 1e6:7.1f}M {s['params_trainable'] / 1e6:9.3f}M "
          f"{s['dev_f1']:7.4f} {s['dev_f1_CL']:6.3f} {s['dev_f1_IT']:6.3f} {s['dev_f1_raw']:6.3f} {s['tau']:4.2f} {s['dev_acc']:6.3f} {s['best_epoch']:3d} "
          f"{s['train_minutes']:5.1f}m {s['peak_gpu_gb']:6.2f} {s['infer_test_seconds']:6.0f}")
if rows:
    print("\ndevF1 = official macro-F1 over all 13 labels (same as evaluate_phase3.py), after prior calibration tau; raw = without")
    print("\nper-label dev F1:")
    short = [l[:9] for l in rows[0]["labels"]]
    print(f"{'run':24s} " + " ".join(f"{x:>9s}" for x in short))
    for s in rows:
        print(f"{s['run']:24s} " + " ".join(f"{s['dev_per_label'][l][2]:9.3f}" for l in s["labels"]))
json.dump(rows, open(f"{WORK}/p3_results.json", "w"), indent=1)
api.upload_file(path_or_fileobj=f"{WORK}/p3_results.json", path_in_repo="phase3/results/p3_results.json",
                repo_id=CKPT_REPO, repo_type="model")
print("\nzips ready in", ZIPS, ":", sorted(os.listdir(ZIPS)))


run                      model   method  ctx          params  trainable   devF1     CL     IT    raw  tau    acc  ep  train GPU GB test s
p3d_llama_lorma          llama   lorma   doc          112.2M     4.477M  0.4987  0.426  0.507  0.486 0.25  0.635  14  11.8m   3.97      4
p3d_llama_lorma_e35_s14  llama   lorma   doc          112.2M     4.477M  0.4866  0.435  0.485  0.487 0.00  0.618  20  17.7m   3.85      4
p3d_llama_lora           llama   lora    doc          112.4M     4.674M  0.4851  0.434  0.493  0.485 0.00  0.642  15  12.9m   4.17      4
p3d_llama_lora_e35       llama   lora    doc          112.4M     4.674M  0.4850  0.453  0.449  0.485 0.00  0.641  24  21.1m   4.17      4
p4_llama_lorma_ffn_lr3e4 llama   lorma   doc          114.3M     6.542M  0.4847  0.422  0.475  0.485 0.00  0.622  12  13.7m   5.19      6
p3d_llama_lorma_e35      llama   lorma   doc          112.2M     4.477M  0.4755  0.416  0.482  0.475 0.00  0.613  17  15.7m   3.97      4
p3_llama_lorma           llama   l

In [9]:
# ===================== P3 CELL 9 - Ensembles (plain top-K averages + greedy) =====================
if torch.cuda.device_count():
    r = subprocess.run([sys.executable, f"{WORK}/p3ens.py", "--data_dir", RAW, "--tok", TOK_PATH, "--out_dir", OUT,
                        "--hf_repo", CKPT_REPO], env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"), capture_output=True, text=True)
    print("\n".join([l for l in (r.stdout + r.stderr).splitlines() if "[ensemble]" in l or "Error" in l or "Traceback" in l][-40:]))
    if r.returncode != 0:
        print("\n".join((r.stdout + r.stderr).splitlines()[-30:]))
    for o in ["ens_top3", "ens_top5", "ens_top8", "ens_greedy", "ens_top3_seq", "ens_top5_seq", "ens_top8_seq", "ens_greedy_seq", "best1_seq"]:
        if os.path.exists(f"{OUT}/{o}/submission.zip"):
            shutil.copy(f"{OUT}/{o}/submission.zip", f"{ZIPS}/{o}.zip")
    print("zips:", sorted(os.listdir(ZIPS)))
else:
    print("CPU session: ensembles skipped")


[05:21:38][ensemble] 33 candidate runs: best1_seq=0.5070, p3d_llama_lorma=0.4987, p3d_llama_lorma_e35_s14=0.4866, p3d_llama_lora=0.4851, p3d_llama_lora_e35=0.4850, p4_llama_lorma_ffn_lr3e4=0.4847, p3d_llama_lorma_e35=0.4755, p3_llama_lorma=0.4732, p3d_llama_ia3=0.4706, p4_llama_lorma_ffn=0.4704, p3d_llama_full=0.4608, p4_llama_lora_ffn=0.4602, p4_llama_lorma_ffn_ls=0.4524, p4_llama_lorma_ffn_s14=0.4518, p3_gpt2_lorma=0.4490, p3d_gpt2_lorma=0.4480, p3_llama_lorma_win=0.4431, p3_llama_lora=0.4408, p3_llama_rslora=0.4259, p3_llama_ia3=0.4241, p3_llama_full=0.4224, p3_gpt2_full=0.4172, p3d_shared_full=0.4121, p3_shared_kd=0.4069, p3_shared_full=0.3877, p4_gpt2_lorma_ffn=0.3875, p3d_shared_kd=0.3799, p3d_gpt2_lorma_e35=0.3568, p3d_encdec_lorma=0.3416, p3_llama_head=0.3307, p3_encdec_lora=0.3110, p3_encdec_lorma=0.3096, p3_llama_lorma_sent=0.2434
Traceback (most recent call last):
AttributeError: 'NoneType' object has no attribute 'seek'
Traceback (most recent call last):
AttributeError: 'No